# WSJ 2027 - Scoutnet Participant Map

Interactive KeplerGL visualization of Swedish scout participants registered for World Scout Jamboree 2027.

**Data source:** Scoutnet API  
**Visualization:** KeplerGL with heatmaps and point layers  
**Colors:** Scout Blue (#004B87) for deltagare, Gold (#E8A838) for IST

In [1]:
# Cell 1: API Configuration
import requests
import json
import os
import pandas as pd
from collections import Counter, defaultdict
from datetime import date

# Load credentials from scoutnet_secrets.py (gitignored)
from scoutnet_secrets import SCOUTNET_API_ID, SCOUTNET_API_KEY

# WSJ 2027 age cutoffs
# Deltagare: age 14-17 at event (born 2010-2013 roughly)
# IST: age 18+ at event
WSJ_DATE = date(2027, 8, 1)

# Scoutnet API endpoint (project participants)
SCOUTNET_URL = "https://www.scoutnet.se/api/project/get/participants"

print("API configuration loaded")

API configuration loaded


In [16]:
# Cell 2: Fetch participant list from Scoutnet API

def fetch_scoutnet_participants():
    """Fetch all participants from the Scoutnet project."""
    response = requests.get(
        SCOUTNET_URL,
        auth=(SCOUTNET_API_ID, SCOUTNET_API_KEY)
    )
    response.raise_for_status()
    
    data = response.json()
    participants = data.get('participants', {})
    labels = data.get('labels', {})
    
    print(f"Fetched {len(participants)} participants from Scoutnet API")
    
    # Show fee types (registration categories)
    fee_labels = labels.get('project_fee', {})
    print(f"\nFee categories:")
    for fee_id, label in fee_labels.items():
        if label:
            count = sum(1 for p in participants.values() if str(p.get('fee_id', '')) == fee_id)
            print(f"  {fee_id}: {label} ({count} st)")
    
    # Count status
    cancelled = sum(1 for p in participants.values() if p.get('cancelled'))
    confirmed = sum(1 for p in participants.values() if p.get('confirmed') and not p.get('cancelled'))
    unconfirmed = sum(1 for p in participants.values() if not p.get('confirmed') and not p.get('cancelled'))
    print(f"\nConfirmed: {confirmed}")
    print(f"Unconfirmed: {unconfirmed}")
    print(f"Cancelled: {cancelled}")
    print(f"Active (confirmed only): {confirmed}")
    
    return data

members_data = fetch_scoutnet_participants()

Task was destroyed but it is pending!
task: <Task pending name='Task-244' coro=<_async_in_context.<locals>.run_in_context() done, defined at /usr/local/lib/python3.13/dist-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-245' coro=<Kernel.shell_main() running at /usr/local/lib/python3.13/dist-packages/ipykernel/kernelbase.py:590> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /usr/local/lib/python3.13/dist-packages/zmq/eventloop/zmqstream.py:563]>
/usr/lib/python3.13/json/decoder.py:361: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  obj, end = self.scan_once(s, idx)
Task was destroyed but it is pending!
task: <Task pending name='Task-245' coro=<Kernel.shell_main() running at /usr/local/lib/python3.13/dist-packages/ipykernel/kernelbase.py:590> cb=[Task.task_wakeup()]>


Fetched 1225 participants from Scoutnet API

Fee categories:
  27561: Deltagare direktresa ansökningsavgift (206 st)
  25702: IST egen resa  ansökningsavgift (128 st)
  25694: Deltagare rundresa ansökningsavgift (808 st)
  25697: CMT betalning 1 (21 st)
  25696: IST rundresa ansökningsavgift (56 st)
  25693: Anställd/Styrelse (1 st)

Confirmed: 1166
Unconfirmed: 27
Cancelled: 32
Active (confirmed only): 1166


In [3]:
# Cell 3: Categorize participants (deltagare vs IST vs CMT, rundresa vs direkt) by kår
# Fee IDs from API labels:
#   27561 = Deltagare direktresa ansökningsavgift
#   25694 = Deltagare rundresa ansökningsavgift
#   25702 = IST egen resa ansökningsavgift
#   25696 = IST rundresa ansökningsavgift
#   25697 = CMT betalning 1
#   25693 = Anställd/Styrelse

DELTAGARE_RUNDRESA = '25694'
DELTAGARE_DIREKTRESA = '27561'
DELTAGARE_FEES = {DELTAGARE_RUNDRESA, DELTAGARE_DIREKTRESA}

IST_RUNDRESA = '25696'
IST_EGEN_RESA = '25702'
IST_FEES = {IST_RUNDRESA, IST_EGEN_RESA}

CMT_FEES = {'25697', '25693'}  # CMT + Anställd/Styrelse

# WSJ 2027 Poland: July 29 - August 8, 2027
WSJ_START = date(2027, 7, 29)

def exact_age(birth_date, ref_date):
    """Calculate exact age in years at ref_date."""
    years = ref_date.year - birth_date.year
    if (ref_date.month, ref_date.day) < (birth_date.month, birth_date.day):
        years -= 1
    return years

def categorize_participants(data):
    """Parse participants and categorize by kår into deltagare/IST/CMT and rundresa/direkt.
    Only includes confirmed participants. Validates exact age at WSJ start."""
    karer = defaultdict(lambda: {
        'deltagare_rundresa': 0, 'deltagare_direkt': 0,
        'ist_rundresa': 0, 'ist_direkt': 0,
        'cmt': 0
    })
    
    participants = data.get('participants', {})
    total = 0
    skipped_cancelled = 0
    skipped_unconfirmed = 0
    skipped_wrong_age = []
    no_kar_count = 0
    
    for member_id, p in participants.items():
        if p.get('cancelled'):
            skipped_cancelled += 1
            continue
        if not p.get('confirmed'):
            skipped_unconfirmed += 1
            continue
        
        membership = p.get('primary_membership_info', {})
        if isinstance(membership, dict):
            kar_name = membership.get('group_name', '')
        else:
            kar_name = ''
        
        if not kar_name:
            no_kar_count += 1
        
        fee_id = str(p.get('fee_id', ''))
        dob = p.get('date_of_birth', '')
        name = f"{p.get('first_name', '')} {p.get('last_name', '')}"
        
        birth = None
        if dob:
            try:
                birth = date.fromisoformat(dob)
            except ValueError:
                pass
        
        if fee_id in CMT_FEES:
            if kar_name:
                karer[kar_name]['cmt'] += 1
            total += 1
        elif fee_id in DELTAGARE_FEES:
            if birth is not None:
                age = exact_age(birth, WSJ_START)
                if age < 14 or age >= 18:
                    skipped_wrong_age.append(
                        f"  DELTAGARE wrong age: {name} ({kar_name or 'ingen kår'}) "
                        f"born {dob} (age {age} at WSJ) - must be 14-17")
                    continue
            if kar_name:
                if fee_id == DELTAGARE_RUNDRESA:
                    karer[kar_name]['deltagare_rundresa'] += 1
                else:
                    karer[kar_name]['deltagare_direkt'] += 1
            total += 1
        elif fee_id in IST_FEES:
            if birth is not None:
                age = exact_age(birth, WSJ_START)
                if age < 18:
                    skipped_wrong_age.append(
                        f"  IST too young: {name} ({kar_name or 'ingen kår'}) "
                        f"born {dob} (age {age} at WSJ) - must be 18+")
                    continue
            if kar_name:
                if fee_id == IST_RUNDRESA:
                    karer[kar_name]['ist_rundresa'] += 1
                else:
                    karer[kar_name]['ist_direkt'] += 1
            total += 1
        else:
            if birth is not None:
                age = exact_age(birth, WSJ_START)
                if 14 <= age <= 17:
                    if kar_name:
                        karer[kar_name]['deltagare_direkt'] += 1
                    total += 1
                elif age >= 18:
                    if kar_name:
                        karer[kar_name]['ist_direkt'] += 1
                    total += 1
                else:
                    skipped_wrong_age.append(
                        f"  UNKNOWN: {name} ({kar_name or 'ingen kår'}) fee={fee_id} age {age}")
                    continue
            else:
                skipped_wrong_age.append(
                    f"  NO DOB: {name} ({kar_name or 'ingen kår'}) fee={fee_id}")
                continue
    
    print(f"Processed {total} confirmed participants across {len(karer)} kårer")
    print(f"Skipped: {skipped_cancelled} cancelled, {skipped_unconfirmed} unconfirmed")
    print(f"Participants without kår (included in count, not on map): {no_kar_count}")
    
    if skipped_wrong_age:
        print(f"\nSkipped {len(skipped_wrong_age)} with wrong age:")
        for msg in skipped_wrong_age:
            print(msg)
    else:
        print("\nNo age validation issues found")
    
    return dict(karer)

karer_data = categorize_participants(members_data)

Processed 1153 confirmed participants across 278 kårer
Skipped: 33 cancelled, 27 unconfirmed
Participants without kår (included in count, not on map): 4

No age validation issues found


In [4]:
# Cell 4: Summary statistics
del_rund = sum(v['deltagare_rundresa'] for v in karer_data.values())
del_dir = sum(v['deltagare_direkt'] for v in karer_data.values())
ist_rund = sum(v['ist_rundresa'] for v in karer_data.values())
ist_dir = sum(v['ist_direkt'] for v in karer_data.values())
cmt_total = sum(v['cmt'] for v in karer_data.values())
total_del = del_rund + del_dir
total_ist = ist_rund + ist_dir
total_all = total_del + total_ist + cmt_total

print(f"=== WSJ 2027 Registration Statistics ===")
print(f"Total registrerade: {total_all}")
print(f"  Deltagare (14-17): {total_del}")
print(f"    - Rundresa:      {del_rund}")
print(f"    - Direktresa:    {del_dir}")
print(f"  IST (18+):         {total_ist}")
print(f"    - Rundresa:      {ist_rund}")
print(f"    - Egen resa:     {ist_dir}")
print(f"  CMT:               {cmt_total}")
print(f"Antal kårer:         {len(karer_data)}")

=== WSJ 2027 Registration Statistics ===
Total registrerade: 1149
  Deltagare (14-17): 964
    - Rundresa:      775
    - Direktresa:    189
  IST (18+):         163
    - Rundresa:      48
    - Egen resa:     115
  CMT:               22
Antal kårer:         278


In [5]:
# Cell 5: Top kårer by deltagare
top_deltagare = sorted(karer_data.items(),
    key=lambda x: x[1]['deltagare_rundresa'] + x[1]['deltagare_direkt'], reverse=True)[:15]

print("=== Top 15 kårer (deltagare) ===")
for kar, d in top_deltagare:
    total = d['deltagare_rundresa'] + d['deltagare_direkt']
    ist = d['ist_rundresa'] + d['ist_direkt']
    print(f"  {kar}: {total} deltagare ({d['deltagare_rundresa']} rund, {d['deltagare_direkt']} direkt), {ist} IST")

=== Top 15 kårer (deltagare) ===
  Mölndals Scoutkår: 17 deltagare (16 rund, 1 direkt), 3 IST
  Älvsjö Scoutkår: 17 deltagare (7 rund, 10 direkt), 1 IST
  Saltsjö-Boo Scoutkår: 17 deltagare (14 rund, 3 direkt), 0 IST
  Rydebäcks Scoutkår: 16 deltagare (16 rund, 0 direkt), 1 IST
  Scoutkåren Finn: 16 deltagare (14 rund, 2 direkt), 0 IST
  Årsta Scoutkår: 15 deltagare (15 rund, 0 direkt), 1 IST
  Karlsro Scoutkår: 14 deltagare (13 rund, 1 direkt), 4 IST
  Dalby Scoutkår: 14 deltagare (12 rund, 2 direkt), 0 IST
  Vallentuna Scoutkår: 14 deltagare (11 rund, 3 direkt), 4 IST
  Equmenia Scout: 13 deltagare (10 rund, 3 direkt), 5 IST
  Örnsbergs Scoutkår: 13 deltagare (10 rund, 3 direkt), 4 IST
  Segeltorps Scoutkår: 13 deltagare (10 rund, 3 direkt), 2 IST
  Nyköpings Scoutkår: 12 deltagare (12 rund, 0 direkt), 2 IST
  Staffanstorps Scoutkår Torparna: 11 deltagare (11 rund, 0 direkt), 0 IST
  Furulunds Scoutkår: 11 deltagare (10 rund, 1 direkt), 0 IST


In [6]:
# Cell 6: Top kårer by IST
top_ist = sorted(karer_data.items(),
    key=lambda x: x[1]['ist_rundresa'] + x[1]['ist_direkt'], reverse=True)[:15]

print("=== Top 15 kårer (IST) ===")
for kar, d in top_ist:
    ist = d['ist_rundresa'] + d['ist_direkt']
    total_del = d['deltagare_rundresa'] + d['deltagare_direkt']
    print(f"  {kar}: {ist} IST ({d['ist_rundresa']} rund, {d['ist_direkt']} egen), {total_del} deltagare")

=== Top 15 kårer (IST) ===
  Karlstads Scoutkår: 7 IST (0 rund, 7 egen), 2 deltagare
  Equmenia Scout: 5 IST (3 rund, 2 egen), 13 deltagare
  Karlsro Scoutkår: 4 IST (1 rund, 3 egen), 14 deltagare
  Örnsbergs Scoutkår: 4 IST (0 rund, 4 egen), 13 deltagare
  Bollstanäs Scoutkår: 4 IST (4 rund, 0 egen), 7 deltagare
  Vallentuna Scoutkår: 4 IST (0 rund, 4 egen), 14 deltagare
  S:t Olofs Scoutkår Västerås: 4 IST (1 rund, 3 egen), 5 deltagare
  Vässarö Scoutkår: 3 IST (0 rund, 3 egen), 0 deltagare
  Landskrona KM: 3 IST (1 rund, 2 egen), 2 deltagare
  Mölndals Scoutkår: 3 IST (0 rund, 3 egen), 17 deltagare
  Wä Scoutkår: 3 IST (0 rund, 3 egen), 4 deltagare
  Kullavik Sjöscoutkår: 3 IST (0 rund, 3 egen), 0 deltagare
  Saltsjöbadens Sjöscoutkår: 3 IST (0 rund, 3 egen), 1 deltagare
  Tornugglan Scoutkår: 3 IST (0 rund, 3 egen), 4 deltagare
  Handens Scoutkår: 2 IST (0 rund, 2 egen), 0 deltagare


In [7]:
# Cell 7: Distribution histogram
print("=== Distribution of deltagare per kår ===")
deltagare_counts = [v['deltagare_rundresa'] + v['deltagare_direkt']
                    for v in karer_data.values()
                    if v['deltagare_rundresa'] + v['deltagare_direkt'] > 0]

for bucket in range(0, max(deltagare_counts) + 2, 2):
    count = sum(1 for d in deltagare_counts if bucket <= d < bucket + 2)
    if count > 0:
        print(f"  {bucket:2d}-{bucket+1:2d} deltagare: {'█' * count} ({count} kårer)")

print(f"\nKårer med bara IST (0 deltagare): {sum(1 for v in karer_data.values() if v['deltagare_rundresa'] + v['deltagare_direkt'] == 0 and v['ist_rundresa'] + v['ist_direkt'] > 0)}")
print(f"Kårer med bara deltagare (0 IST): {sum(1 for v in karer_data.values() if v['ist_rundresa'] + v['ist_direkt'] == 0 and v['deltagare_rundresa'] + v['deltagare_direkt'] > 0)}")

=== Distribution of deltagare per kår ===
   0- 1 deltagare: ███████████████████████████████████████████████████████████████████████████ (75 kårer)
   2- 3 deltagare: ██████████████████████████████████████████████████████████████████████████████ (78 kårer)
   4- 5 deltagare: █████████████████████████████████████████ (41 kårer)
   6- 7 deltagare: ████████████████████ (20 kårer)
   8- 9 deltagare: ████████████ (12 kårer)
  10-11 deltagare: █████████ (9 kårer)
  12-13 deltagare: ████ (4 kårer)
  14-15 deltagare: ████ (4 kårer)
  16-17 deltagare: █████ (5 kårer)

Kårer med bara IST (0 deltagare): 26
Kårer med bara deltagare (0 IST): 172


In [8]:
# Cell 8: Geocoding with cache + manual corrections
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import time

CACHE_FILE = '/config/notebooks/wsj27/scoutkar_geocode_cache.json'

# Manual corrections for kårer that geocode to wrong locations
MANUAL_CORRECTIONS = {
    'Landeryd Scoutkår': (58.3725, 15.7001),
    'Staffanstorps Scoutkår': (55.6419, 13.2063),
    'Gustaf Vasa-Bredängs Scoutkår': (59.2941, 17.9383),
    'Mälarscouternas Scoutkår': (59.3326, 18.0649),
    'Sköndals Scoutkår': (59.2571, 18.1071),
    'Blackebergs Scoutkår': (59.3500, 17.8833),
    'S:t Görans Scoutkår': (59.3333, 18.0333),
    'Hässelby Strands Scoutkår': (59.3667, 17.8333),
    'Staffanstorps Scoutkår Torparna': (55.6419, 13.2063),
    'Scoutkåren Gustaf Vasa-Bredäng': (59.2941, 17.9383),
    'Husie-Hohögs Scoutkår': (55.5833, 13.1000),
    'Trekungakåren': (57.7089, 11.9746),
    'Scoutkåren Vikingen-Ekeby': (59.3667, 16.5000),
    'Näsby-Österslövs scoutkår': (56.1500, 14.1500),
    'Scoutkåren Mälarscouterna': (59.3326, 18.0649),
    'Sjöscoutkåren Drakarna': (59.3293, 18.0686),
    'Scoutkåren Vikingarna': (59.3293, 18.0686),
    'Drottningstaden Scoutkår': (58.5372, 13.1500),
    'Östra Eneby-Haga Scoutkår': (58.6167, 16.1833),
    'Eldsberga-Tönnersjö Scoutkår': (56.7167, 12.9500),
    'Tornugglan Scoutkår': (55.7119, 13.2283),
    'Lutherska Missionskyrkans EFS Scout (Salt) Toleredskyrkans Scoutkår': (57.7167, 11.9333),
    'Wä Scoutkår': (56.0167, 14.2167),
    'Scoutkåren Peter Momma': (59.2000, 17.8500),
    'Sjöscoutkåren S:t Göran': (59.3333, 18.0333),
    'Blackebergs sjöscoutkår': (59.3500, 17.8833),
    'Sköndals Land- och Sjöscoutkår': (59.2571, 18.1071),
    'Malå Equmenia': (65.1833, 18.7500),
    'Turinge Scoutkår': (59.1500, 17.5667),
}

WEBSEARCH_LOCATIONS = {
    'Annestorpsdalens Scoutkår': (57.6523, 12.0712),
    'Brobergska Scoutkår': (59.3293, 18.0686),
    'S:t Örjans Scoutkår': (57.7210, 12.9401),
    'Finns Scoutkår': (55.7047, 13.1910),
    'Tornugglans Scoutkår': (55.7119, 13.2283),
    'Redbergslids Scoutkår': (57.7089, 12.0000),
    'Göta Lejons Scoutkår': (57.7089, 11.9746),
    'Jägarna-Lundhagskyrkans Scoutkår': (59.8607, 17.6253),
    'Vikingarna Scoutkår': (59.3293, 18.0686),
    'Drakarna Scoutkår': (59.3293, 18.0686),
    'Equmeniascout Lerum': (57.7700, 12.2692),
    'Södermöre Scoutkår': (56.6833, 16.3667),
    'Vikingen-Ekeby Scoutkår': (59.3667, 16.5000),
    'Peter Momma Scoutkår': (59.2000, 17.8500),
    'Drottens Scoutkår': (58.4108, 15.6214),
    'Lyckebokyrkans Scoutkår': (57.7833, 12.0500),
    'Toleredskyrkans Scoutkår': (57.7167, 11.9333),
    'Lundhagskyrkans Scoutkår': (59.8560, 17.6097),
}

# Load cache
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        geocode_cache = json.load(f)
    print(f"Loaded {len(geocode_cache)} cached geocode entries")
else:
    geocode_cache = {}
    print("No cache file found, starting fresh")

# Apply manual corrections (override cache)
for kar_name in karer_data.keys():
    if kar_name in MANUAL_CORRECTIONS:
        lat, lng = MANUAL_CORRECTIONS[kar_name]
        geocode_cache[kar_name] = {'lat': lat, 'lng': lng, 'source': 'manual'}
    elif kar_name in WEBSEARCH_LOCATIONS:
        lat, lng = WEBSEARCH_LOCATIONS[kar_name]
        geocode_cache[kar_name] = {'lat': lat, 'lng': lng, 'source': 'websearch'}

# Geocode missing kårer
geolocator = Nominatim(user_agent="wsj2027_scout_map")
new_geocodes = 0

for kar_name in karer_data.keys():
    if kar_name in geocode_cache:
        continue
    if kar_name == 'Gäst / Annat':
        continue
    
    try:
        search_name = kar_name
        for remove in ['Sjöscoutkår', 'Sjöscoutkåren', 'Scoutkåren ', 'Scoutkår',
                        'Kustscoutkår', 'Spanarkår', 'Distriktskår', 'scoutkår',
                        'KFUM ', 'KFUM', 'Equmenia ', 'Equmeniascout ']:
            search_name = search_name.replace(remove, '')
        search_name = search_name.strip(' -')
        
        location = geolocator.geocode(f"{search_name}, Sweden", timeout=10)
        
        if location and 55 < location.latitude < 70 and 10 < location.longitude < 25:
            geocode_cache[kar_name] = {
                'lat': location.latitude, 'lng': location.longitude,
                'source': 'nominatim', 'display': location.address
            }
            new_geocodes += 1
            print(f"  Geocoded: {kar_name} -> {location.latitude:.4f}, {location.longitude:.4f}")
        else:
            parts = search_name.split()
            if len(parts) > 1:
                location = geolocator.geocode(f"{parts[-1]}, Sweden", timeout=10)
                time.sleep(1.1)
                if location and 55 < location.latitude < 70:
                    geocode_cache[kar_name] = {
                        'lat': location.latitude, 'lng': location.longitude,
                        'source': 'nominatim_fallback', 'display': location.address
                    }
                    new_geocodes += 1
                    print(f"  Geocoded (fallback): {kar_name} -> {location.latitude:.4f}")
                else:
                    print(f"  FAILED: {kar_name}")
                    geocode_cache[kar_name] = {'lat': None, 'lng': None, 'source': 'failed'}
            else:
                print(f"  FAILED: {kar_name}")
                geocode_cache[kar_name] = {'lat': None, 'lng': None, 'source': 'failed'}
        
        time.sleep(1.1)
        if new_geocodes % 20 == 0 and new_geocodes > 0:
            with open(CACHE_FILE, 'w', encoding='utf-8') as f:
                json.dump(geocode_cache, f, ensure_ascii=False, indent=2)
    except GeocoderTimedOut:
        print(f"  TIMEOUT: {kar_name}")
        geocode_cache[kar_name] = {'lat': None, 'lng': None, 'source': 'timeout'}
    except Exception as e:
        print(f"  ERROR: {kar_name}: {e}")
        geocode_cache[kar_name] = {'lat': None, 'lng': None, 'source': f'error: {e}'}

with open(CACHE_FILE, 'w', encoding='utf-8') as f:
    json.dump(geocode_cache, f, ensure_ascii=False, indent=2)

print(f"\nGeocoded {new_geocodes} new kårer")
print(f"Total cached: {len(geocode_cache)}")

# Build DataFrame with rundresa/direkt split + CMT
rows = []
failed = []
for kar_name, counts in karer_data.items():
    if kar_name == 'Gäst / Annat':
        continue
    geo = geocode_cache.get(kar_name, {})
    lat = geo.get('lat')
    lng = geo.get('lng')
    
    if lat is not None and lng is not None:
        del_total = counts['deltagare_rundresa'] + counts['deltagare_direkt']
        ist_total = counts['ist_rundresa'] + counts['ist_direkt']
        cmt_count = counts.get('cmt', 0)
        rows.append({
            'kar': kar_name,
            'lat': lat,
            'lng': lng,
            'antal': del_total + ist_total + cmt_count,
            'deltagare': del_total,
            'deltagare_rundresa': counts['deltagare_rundresa'],
            'deltagare_direkt': counts['deltagare_direkt'],
            'ist': ist_total,
            'ist_rundresa': counts['ist_rundresa'],
            'ist_direkt': counts['ist_direkt'],
            'cmt': cmt_count,
        })
    else:
        failed.append(kar_name)

df_map = pd.DataFrame(rows)
print(f"\n{len(df_map)} kårer with coordinates")
print(f"{df_map['antal'].sum()} participants mapped ({df_map['antal'].sum() / total_all * 100:.1f}%)")

if failed:
    print(f"\n{len(failed)} kårer without coordinates:")
    for f_name in failed:
        d = karer_data[f_name]
        t = d['deltagare_rundresa'] + d['deltagare_direkt']
        i = d['ist_rundresa'] + d['ist_direkt']
        c = d.get('cmt', 0)
        print(f"  - {f_name} ({t} del, {i} ist, {c} cmt)")

Loaded 284 cached geocode entries

Geocoded 0 new kårer
Total cached: 284

277 kårer with coordinates
1148 participants mapped (99.9%)

1 kårer without coordinates:
  - Lidingö-Bodals Sjöscoutkår (1 del, 0 ist, 0 cmt)


In [ ]:
# Cell 9: Prepare KeplerGL map config
# Map generated directly from CDN (no keplergl Python package needed)
# One point per kår. Size = total count. Tooltip shows rundresa/direkt breakdown + CMT.
# Colors: Scout Blue (#004B87) for deltagare, Gold (#E8A838) for IST

kepler_config = {
    'version': 'v1',
    'config': {
        'mapState': {
            'latitude': 59.0,
            'longitude': 16.0,
            'zoom': 5
        },
        'visState': {
            'interactionConfig': {
                'tooltip': {
                    'fieldsToShow': {
                        'karer': [
                            {'name': 'kar', 'format': None},
                            {'name': 'deltagare_rundresa', 'format': None},
                            {'name': 'deltagare_direkt', 'format': None},
                            {'name': 'ist_rundresa', 'format': None},
                            {'name': 'ist_direkt', 'format': None},
                            {'name': 'cmt', 'format': None},
                            {'name': 'antal', 'format': None},
                        ]
                    },
                    'enabled': True
                }
            },
            'layers': [
                {
                    'id': 'deltagare-points',
                    'type': 'point',
                    'config': {
                        'dataId': 'karer',
                        'label': 'Deltagare',
                        'isVisible': True,
                        'columns': {'lat': 'lat', 'lng': 'lng'},
                        'color': [0, 75, 135],
                        'sizeField': {'name': 'deltagare', 'type': 'integer'},
                        'sizeScale': 'sqrt',
                        'visConfig': {
                            'radius': 10,
                            'fixedRadius': False,
                            'opacity': 0.85,
                            'outline': True,
                            'thickness': 1,
                            'strokeColor': [255, 255, 255],
                            'radiusRange': [4, 30],
                            'filled': True
                        }
                    }
                },
                {
                    'id': 'ist-points',
                    'type': 'point',
                    'config': {
                        'dataId': 'karer',
                        'label': 'IST',
                        'isVisible': True,
                        'columns': {'lat': 'lat', 'lng': 'lng'},
                        'color': [232, 168, 56],
                        'sizeField': {'name': 'ist', 'type': 'integer'},
                        'sizeScale': 'sqrt',
                        'visConfig': {
                            'radius': 8,
                            'fixedRadius': False,
                            'opacity': 0.9,
                            'outline': True,
                            'thickness': 1,
                            'strokeColor': [255, 255, 255],
                            'radiusRange': [3, 30],
                            'filled': True
                        }
                    }
                }
            ]
        }
    }
}

# Prepare data in KeplerGL DataFrame format
data_config = {
    'config': kepler_config,
    'data': {
        'karer': df_map.to_dict(orient='split')
    },
    'options': {'readOnly': False, 'centerMap': False}
}

print(f"Map config ready: {len(df_map)} kårer, {len(kepler_config['config']['visState']['layers'])} layers")

In [ ]:
# Cell 10: Generate KeplerGL HTML directly (no Python keplergl package needed)
# Uses kepler.gl@2.5.5 from CDN with default component (full UI including export)
OUTPUT_DIR = '/config/notebooks/wsj27'

try:
    from scoutnet_secrets import MAPBOX_TOKEN
except ImportError:
    MAPBOX_TOKEN = os.environ.get("MAPBOX_TOKEN", "")

HTML_TEMPLATE = '''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>WSJ 2027 Scout Map</title>
<link href="https://d1a3f4spazzrp4.cloudfront.net/kepler.gl/uber-fonts/4.0.0/superfine.css" rel="stylesheet">
<link href="https://api.tiles.mapbox.com/mapbox-gl-js/v1.1.1/mapbox-gl.css" rel="stylesheet">
<script src="https://unpkg.com/react@17.0.2/umd/react.production.min.js" crossorigin></script>
<script src="https://unpkg.com/react-dom@17.0.2/umd/react-dom.production.min.js" crossorigin></script>
<script src="https://unpkg.com/redux@3.7.2/dist/redux.js" crossorigin></script>
<script src="https://unpkg.com/react-redux@7.1.3/dist/react-redux.min.js" crossorigin></script>
<script src="https://unpkg.com/react-intl@3.12.0/dist/react-intl.min.js" crossorigin></script>
<script src="https://unpkg.com/react-copy-to-clipboard@5.0.2/build/react-copy-to-clipboard.min.js" crossorigin></script>
<script src="https://unpkg.com/styled-components@4.1.3/dist/styled-components.min.js" crossorigin></script>
<script src="https://unpkg.com/kepler.gl@2.5.5/umd/keplergl.min.js" crossorigin></script>
<style>
  font-family: ff-clan-web-pro, 'Helvetica Neue', Helvetica, sans-serif;
  font-weight: 400;
  font-size: 0.875em;
  line-height: 1.71429;
  *, *:before, *:after { box-sizing: border-box; }
  body { margin: 0; padding: 0; }
  #app { width: 100vw; height: 100vh; position: absolute; top: 0; left: 0; }
</style>
</head>
<body>
<div id="app"></div>
<script>window.__keplerglDataConfig = __DATA_CONFIG__;</script>
<script>
(function() {
  var K = window.KeplerGl;
  var Redux = window.Redux;

  // Store with KeplerGL reducer
  var reducer = K.keplerGlReducer.initialState({
    uiState: { currentModal: null, activeSidePanel: null }
  });
  var middlewares = K.enhanceReduxMiddleware([]);
  var store = Redux.createStore(
    Redux.combineReducers({ keplerGl: reducer }),
    {},
    Redux.applyMiddleware.apply(null, middlewares)
  );

  // Default KeplerGL component with all features (export, filters, etc)
  var MapApp = K.injectComponents([]);

  // Resize tracking
  var dims = { w: window.innerWidth, h: window.innerHeight };
  function App() {
    var ref = React.useState(dims);
    var size = ref[0], setSize = ref[1];
    React.useEffect(function() {
      function onResize() { setSize({ w: window.innerWidth, h: window.innerHeight }); }
      window.addEventListener('resize', onResize);
      return function() { window.removeEventListener('resize', onResize); };
    }, []);
    return React.createElement(
      ReactRedux.Provider, { store: store },
      React.createElement(MapApp, {
        mapboxApiAccessToken: '__MAPBOX_TOKEN__',
        id: 'map',
        width: size.w,
        height: size.h,
        appName: 'WSJ 2027'
      })
    );
  }

  ReactDOM.render(React.createElement(App), document.getElementById('app'));

  // Load data
  var cfg = window.__keplerglDataConfig;
  var datasets = Object.keys(cfg.data).map(function(key) {
    var d = cfg.data[key];
    return {
      info: { id: key, label: key },
      data: {
        fields: d.columns.map(function(c) { return { name: c }; }),
        rows: d.data
      }
    };
  });

  store.dispatch(K.addDataToMap({
    datasets: datasets,
    config: cfg.config,
    options: cfg.options || { centerMap: true }
  }));
})();
</script>
</body>
</html>'''

# Generate HTML
data_config_json = json.dumps(data_config)
html_content = (HTML_TEMPLATE
    .replace('__DATA_CONFIG__', data_config_json)
    .replace('__MAPBOX_TOKEN__', MAPBOX_TOKEN))

# Save both versions
notebook_html = f'{OUTPUT_DIR}/wsj_keplergl_notebook.html'
with open(notebook_html, 'w', encoding='utf-8') as f:
    f.write(html_content)

fullscreen_html = f'{OUTPUT_DIR}/wsj_keplergl_karta.html'
with open(fullscreen_html, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"Saved: {notebook_html}")
print(f"Saved: {fullscreen_html}")
print(f"\n=== Final Statistics ===")
print(f"Kårer on map: {len(df_map)}")
print(f"Deltagare rundresa:  {df_map['deltagare_rundresa'].sum()}")
print(f"Deltagare direktresa: {df_map['deltagare_direkt'].sum()}")
print(f"IST rundresa:        {df_map['ist_rundresa'].sum()}")
print(f"IST egen resa:       {df_map['ist_direkt'].sum()}")
print(f"CMT:                 {df_map['cmt'].sum()}")
print(f"Total mapped:        {df_map['antal'].sum()}")
print(f"Coverage:            {df_map['antal'].sum() / total_all * 100:.1f}%")

In [11]:
# Cell 11: Linköpings kårer - deltagare, IST och CMT
# Kårer inom Linköpings kommun inkl. Vreta Kloster och Linghem
# Expanded bounds: lat 58.2-58.55, lng 15.4-15.8

LINKOPING_BOUNDS = {'lat_min': 58.2, 'lat_max': 58.55, 'lng_min': 15.4, 'lng_max': 15.8}

# Find Linköping kårer from geocode cache
lkpg_karer = set()
for kar_name, geo in geocode_cache.items():
    lat, lng = geo.get('lat'), geo.get('lng')
    if lat and lng and (LINKOPING_BOUNDS['lat_min'] < lat < LINKOPING_BOUNDS['lat_max']
                        and LINKOPING_BOUNDS['lng_min'] < lng < LINKOPING_BOUNDS['lng_max']):
        lkpg_karer.add(kar_name)

print(f"=== Linköpings kårer ({len(lkpg_karer)} st) ===\n")

# Stats per kår
lkpg_total = {'del_rund': 0, 'del_dir': 0, 'ist_rund': 0, 'ist_dir': 0, 'cmt': 0}
for kar in sorted(lkpg_karer):
    d = karer_data.get(kar, {'deltagare_rundresa': 0, 'deltagare_direkt': 0,
                              'ist_rundresa': 0, 'ist_direkt': 0, 'cmt': 0})
    del_tot = d['deltagare_rundresa'] + d['deltagare_direkt']
    ist_tot = d['ist_rundresa'] + d['ist_direkt']
    cmt_count = d.get('cmt', 0)
    lkpg_total['del_rund'] += d['deltagare_rundresa']
    lkpg_total['del_dir'] += d['deltagare_direkt']
    lkpg_total['ist_rund'] += d['ist_rundresa']
    lkpg_total['ist_dir'] += d['ist_direkt']
    lkpg_total['cmt'] += cmt_count
    parts = [f"{del_tot} deltagare ({d['deltagare_rundresa']} rund, {d['deltagare_direkt']} direkt)"]
    parts.append(f"{ist_tot} IST ({d['ist_rundresa']} rund, {d['ist_direkt']} egen)")
    if cmt_count > 0:
        parts.append(f"{cmt_count} CMT")
    print(f"{kar}: {', '.join(parts)}")

total = (lkpg_total['del_rund'] + lkpg_total['del_dir'] +
         lkpg_total['ist_rund'] + lkpg_total['ist_dir'] + lkpg_total['cmt'])
print(f"\nTotalt Linköping: {total} personer")
print(f"  Deltagare: {lkpg_total['del_rund'] + lkpg_total['del_dir']} ({lkpg_total['del_rund']} rund, {lkpg_total['del_dir']} direkt)")
print(f"  IST:       {lkpg_total['ist_rund'] + lkpg_total['ist_dir']} ({lkpg_total['ist_rund']} rund, {lkpg_total['ist_dir']} egen)")
print(f"  CMT:       {lkpg_total['cmt']}")

# List individual participants
print(f"\n=== Deltagare per kår ===\n")
participants = members_data.get('participants', {})

for kar in sorted(lkpg_karer):
    kar_people = []
    for mid, p in participants.items():
        if p.get('cancelled') or not p.get('confirmed'):
            continue
        membership = p.get('primary_membership_info', {})
        if isinstance(membership, dict) and membership.get('group_name') == kar:
            fee_id = str(p.get('fee_id', ''))
            name = f"{p.get('first_name', '')} {p.get('last_name', '')}"
            dob = p.get('date_of_birth', '')
            birth = None
            if dob:
                try:
                    birth = date.fromisoformat(dob)
                except ValueError:
                    pass
            age = exact_age(birth, WSJ_START) if birth else '?'

            if fee_id in CMT_FEES:
                typ = 'CMT'
                resa = ''
            elif fee_id in DELTAGARE_FEES:
                typ = 'Deltagare'
                resa = 'rundresa' if fee_id == DELTAGARE_RUNDRESA else 'direktresa'
            elif fee_id in IST_FEES:
                typ = 'IST'
                resa = 'rundresa' if fee_id == IST_RUNDRESA else 'egen resa'
            else:
                if isinstance(age, int) and 14 <= age <= 17:
                    typ = 'Deltagare'
                else:
                    typ = 'IST'
                resa = 'okänd'
            kar_people.append((name, typ, resa, age))

    if kar_people:
        print(f"--- {kar} ({len(kar_people)} st) ---")
        for name, typ, resa, age in sorted(kar_people, key=lambda x: (x[1], x[0])):
            resa_str = f", {resa}" if resa else ""
            print(f"  {name} ({typ}{resa_str}, {age} år)")
        print()

=== Linköpings kårer (10 st) ===

Berga Scoutkår: 5 deltagare (5 rund, 0 direkt), 0 IST (0 rund, 0 egen)
Hanekinds scoutkår: 8 deltagare (5 rund, 3 direkt), 1 IST (0 rund, 1 egen)
Johannelunds Scoutkår: 4 deltagare (3 rund, 1 direkt), 0 IST (0 rund, 0 egen)
Kärna Scoutkår Linköping: 4 deltagare (1 rund, 3 direkt), 0 IST (0 rund, 0 egen)
Landeryd Scoutkår: 3 deltagare (3 rund, 0 direkt), 0 IST (0 rund, 0 egen), 2 CMT
Linghems Scoutkår: 3 deltagare (3 rund, 0 direkt), 2 IST (0 rund, 2 egen)
Valla Scoutkår: 4 deltagare (0 rund, 4 direkt), 0 IST (0 rund, 0 egen)
Vist scoutkår: 2 deltagare (2 rund, 0 direkt), 0 IST (0 rund, 0 egen)
Vreta Kloster Scoutkår: 1 deltagare (1 rund, 0 direkt), 0 IST (0 rund, 0 egen)
Vårdnäs Scoutkår: 0 deltagare (0 rund, 0 direkt), 1 IST (0 rund, 1 egen)

Totalt Linköping: 40 personer
  Deltagare: 34 (23 rund, 11 direkt)
  IST:       4 (0 rund, 4 egen)
  CMT:       2

=== Deltagare per kår ===

--- Berga Scoutkår (5 st) ---
  Franz Schmitterblad (Deltagare, rundre